In [1]:
import os
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import json as json_lib 
from pub import Weak, Random, Real

In [2]:
op_dir = '../data/N123/'
data_dir = '../init/N/'
seed = 123
os.makedirs(op_dir, exist_ok=True)

# 1 Data Loading

In [3]:
cols = ['UserId', 'QuestionId', 'IsCorrect', 'AnswerId'] 

train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'), usecols=cols)
valid_df = pd.read_csv(os.path.join(data_dir, 'valid.csv'), usecols=cols)
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'), usecols=cols)
exer_df = pd.read_csv(os.path.join(data_dir, 'q_meta.csv'))

full_df = pd.concat([train_df, valid_df, test_df], ignore_index=True)
print(len(full_df), "\n")

full_df = full_df.sort_values(by='AnswerId')
full_df = full_df.drop_duplicates(subset=['UserId', 'QuestionId'], keep='first')
full_df.rename(columns={'UserId': 'user_id', 
                        'QuestionId': 'problem_id', 
                        'IsCorrect': 'correct', 
                        'AnswerId': 'order_id'}, inplace=True)
print(len(full_df), "\n")
print(full_df.head(), "\n")

exer_df.rename(columns={'QuestionId': 'problem_id', 'SubjectId': 'skill_ids'}, inplace=True)
print(exer_df.head(), "\n")

19834813 

19834813 

          problem_id  user_id  order_id  correct
12482800       22318    80784         0        1
19555639       12358    90224         1        1
9846176        24778    38296         2        0
3809082        10205     9055         3        1
836413         19416    34386         4        1 

   problem_id                            skill_ids
0       13090  [3, 32, 71, 77, 141, 185, 186, 214]
1        1855                 [3, 71, 75, 86, 178]
2       10423                     [3, 32, 38, 239]
3        2290                     [3, 32, 33, 144]
4       12785                     [3, 32, 33, 144] 



In [4]:
def parse_skills(skill_data):
        s = str(skill_data).strip().strip('[]')
        if not s: 
                return []
        return [int(x.strip()) for x in s.split(',')]

exer_df['skill_ids'] = exer_df['skill_ids'].apply(parse_skills)
print(exer_df.head(), "\n")

   problem_id                            skill_ids
0       13090  [3, 32, 71, 77, 141, 185, 186, 214]
1        1855                 [3, 71, 75, 86, 178]
2       10423                     [3, 32, 38, 239]
3        2290                     [3, 32, 33, 144]
4       12785                     [3, 32, 33, 144] 



In [5]:
full_df = pd.merge(full_df, exer_df[['problem_id', 'skill_ids']], on='problem_id', how='inner')
print(full_df.head(), "\n")
full_df = full_df[full_df['skill_ids'].apply(len) > 0]
print(len(full_df), "\n")
print(full_df.head(), "\n")

   problem_id  user_id  order_id  correct          skill_ids
0       22318    80784         0        1   [3, 71, 74, 439]
1       12358    90224         1        1    [3, 49, 62, 70]
2       24778    38296         2        0  [3, 32, 144, 202]
3       10205     9055         3        1  [3, 32, 144, 203]
4       19416    34386         4        1   [3, 71, 98, 209] 

19834813 

   problem_id  user_id  order_id  correct          skill_ids
0       22318    80784         0        1   [3, 71, 74, 439]
1       12358    90224         1        1    [3, 49, 62, 70]
2       24778    38296         2        0  [3, 32, 144, 202]
3       10205     9055         3        1  [3, 32, 144, 203]
4       19416    34386         4        1   [3, 71, 98, 209] 



In [6]:
df = full_df.groupby('user_id').filter(lambda g: len(g) >= 15).copy()
print(len(df), "\n")
print(df.head(), "\n")

19834813 

   problem_id  user_id  order_id  correct          skill_ids
0       22318    80784         0        1   [3, 71, 74, 439]
1       12358    90224         1        1    [3, 49, 62, 70]
2       24778    38296         2        0  [3, 32, 144, 202]
3       10205     9055         3        1  [3, 32, 144, 203]
4       19416    34386         4        1   [3, 71, 98, 209] 



In [7]:
np.random.seed(seed)
uni_us = df['user_id'].unique()
print(uni_us.shape)

if len(uni_us) > 10000:
    sampled_users = np.random.choice(uni_us, size=10000, replace=False)
    df = df[df['user_id'].isin(sampled_users)].copy()
    print(f"Sampled 20000 users. Records: {len(df)}")
else:
    print(f"Using all users. Records: {len(df)}")

(118971,)
Sampled 20000 users. Records: 1672040


In [8]:
s_all = sorted(df['user_id'].unique())
e_all = sorted(df['problem_id'].unique())
k_all = sorted(list(set([k for ks in df['skill_ids'] for k in ks])))

s2idx = {uid: i for i, uid in enumerate(s_all)}
e2idx = {pid: i for i, pid in enumerate(e_all)}
k2idx = {sid: i for i, sid in enumerate(k_all)}

n, m, k = len(s2idx), len(e2idx), len(k2idx)

# 2 ID Mapping

In [9]:
df['user_id'] = df['user_id'].map(s2idx)
df['problem_id'] = df['problem_id'].map(e2idx)
df['skill_ids'] = df['skill_ids'].apply(lambda ks: [k2idx[s] for s in ks])

df.sort_values(by=['user_id', 'order_id'], inplace=True)
df = df.reset_index(drop=True)

print(df[70:80], "\n")

    problem_id  user_id  order_id  correct                         skill_ids
70       21925        0   6364581        1  [0, 1, 9, 17, 30, 190, 195, 196]
71       14728        0   6517115        0                    [0, 1, 5, 165]
72       14592        0   6545146        1                    [0, 1, 7, 312]
73       26847        0   6575964        0                  [0, 17, 29, 118]
74       20204        0   6616789        1                  [0, 17, 32, 104]
75       19768        0   6691211        0                     [0, 1, 3, 96]
76       12909        0   6699014        1                    [0, 1, 6, 164]
77       18463        0   6739342        1                  [0, 39, 45, 131]
78        6735        0   6804697        1                     [0, 1, 9, 15]
79        3790        0   6838197        1                    [0, 1, 8, 175] 



In [10]:
p2k = df.groupby('problem_id')['skill_ids'].first().to_dict()
p2k_to_save = pd.DataFrame(p2k.items(), columns=['problem_id', 'skill_ids'])
p2k_to_save.to_csv(os.path.join(op_dir, 'p2k.csv'), index=False)

In [11]:
n_records = len(df)
avg_k_per_e = np.mean([len(ks) for ks in p2k.values()])
avg_r_per_s = n_records / n
n_correct = df[df['correct'] == 1].shape[0]
n_incorrect = df[df['correct'] == 0].shape[0]

print("\n## Dataset Statistics (NIPS20)")
print(f"- records: {n_records}")
print(f"- students: {n}")
print(f"- questions: {m}")
print(f"- knowledge concepts: {k}")
print(f"- concepts per exercise: {avg_k_per_e:.4f}")
print(f"- records per student: {avg_r_per_s:.2f}")
print(f"- correct records / incorrect records: {n_correct} / {n_incorrect}")


## Dataset Statistics (NIPS20)
- records: 1672040
- students: 10000
- questions: 27599
- knowledge concepts: 388
- concepts per exercise: 4.1710
- records per student: 167.20
- correct records / incorrect records: 1068260 / 603780


In [12]:
meta = {
    'n': int(n), 
    'm': int(m), 
    'k': int(k),
    's2idx': {str(k): int(v) for k, v in s2idx.items()},
    'e2idx': {str(k): int(v) for k, v in e2idx.items()},
    'k2idx': {str(k): int(v) for k, v in k2idx.items()}
}
with open(os.path.join(op_dir, "meta.json"), 'w') as f:
    json.dump(meta, f)

In [13]:
kc_counts_series = pd.Series(index=df.index, dtype=str)

for user_id, group in tqdm(df.groupby('user_id'), "Generating kc_counts"):
    kc_counter = np.zeros(k, dtype=np.int16)
    for index, row in group.iterrows():
        kc_counts_series.at[index] = json_lib.dumps(kc_counter.tolist())
        k_practiced = row['skill_ids']
        if k_practiced:
            kc_counter[k_practiced] += 1
    
df['kc_counts'] = kc_counts_series
df.drop(columns=['skill_ids'], inplace=True)
print(df.head(), "\n")

Generating kc_counts: 100%|██████████| 10000/10000 [01:30<00:00, 110.72it/s]


   problem_id  user_id  order_id  correct  \
0       22570        0      1420        1   
1       25780        0      9172        0   
2       17029        0     39141        1   
3       22821        0     69452        1   
4         780        0    171901        1   

                                           kc_counts  
0  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
1  [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
2  [2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
3  [3, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ...  
4  [4, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ...   



## (1) Weak Coverage Split

In [14]:
weak_splitter = Weak(op_dir, seed)
weak_splitter.split_and_save(df, p2k)


========== 1 Weak-Split (Corrected) ==========


Weak Split: 100%|██████████| 10000/10000 [00:08<00:00, 1222.79it/s]


Weak response proportion: 0.096
Weak Split Done. tv: 1341009, test: 331031


## (2) Random 5-Fold Split

In [15]:
random_splitter = Random(op_dir, n_folds=5, seed=seed)
random_splitter.split_and_save(df, p2k)


========== 2 Random 5-Fold Split ==========


Random Split: 100%|██████████| 10000/10000 [00:04<00:00, 2044.35it/s]


Random 5-Fold Split Done. Avg Weak response proportion: 0.001


## (3) Real Scenario (Time-Series) Split

In [16]:
real_splitter = Real(op_dir=op_dir)
real_splitter.split_and_save(df, p2k)


========== 3 Real-Scenario (Time-Series) Split ==========


Time-Series Split: 100%|██████████| 10000/10000 [00:00<00:00, 23377.55it/s]


Weak response proportion: 0.001
Real Split Done. Train: 999746, Val: 331031, Test: 341263
